# Revenue Forecasting Training Pipeline (Real Current-Year Data)

This notebook implements and documents the production Machine Learning revenue forecasting pipeline for the Finance Automation project.

### Architecture & Guiding Principles
1. **Real Data Only**: Queries strictly from `uploaded_finance_files` and `financial_tb_records` where `file_type = 'tb_current'`, `status = 'processed'`, and `period_year = CURRENT_YEAR`.
2. **Single-Year Time Series**: Excludes previous-year revenue as a required training feature, eliminating reliance on synthetic/mocked past data.
3. **Dynamic File Discovery**: Dynamically resolves the latest processed file for each month (e.g. Jan–Jun 2026 and future months like Jul 2026 upon upload).
4. **Lag-1 Feature Engineering**: Derives `current_revenue_lag_1` strictly from chronological actuals without data leakage.
5. **RandomForestRegressor Baseline**: Evaluated with a strict chronological holdout split.

## 1. Imports & Configuration

In [1]:
import os
import sys
import json
import sqlite3
from pathlib import Path
import pandas as pd
import numpy as np
import joblib
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Resolve backend directory and finance database
NOTEBOOK_DIR = Path(os.getcwd()).resolve()
BACKEND_DIR = NOTEBOOK_DIR.parents[1] if 'notebooks' in str(NOTEBOOK_DIR) else NOTEBOOK_DIR
DB_PATH = BACKEND_DIR / 'finance.db'
MODEL_OUTPUT_PATH = BACKEND_DIR / 'ai' / 'models' / 'revenue_forecasting_model_real_2026.joblib'

print(f'Database Path: {DB_PATH} (Exists: {DB_PATH.exists()})')
print(f'Model Target:  {MODEL_OUTPUT_PATH}')

Cell executed successfully.


## 2. Dynamic Real Current-Year Data Ingestion

In [2]:
MONTH_ORDER = {
    'January': 1, 'February': 2, 'March': 3, 'April': 4, 'May': 5, 'June': 6,
    'July': 7, 'August': 8, 'September': 9, 'October': 10, 'November': 11, 'December': 12
}

conn = sqlite3.connect(f'file:{DB_PATH}?mode=ro', uri=True)

# 1. Discover latest eligible processed current-year files
files_query = '''
    SELECT f.id, f.original_filename, f.period_month, f.period_year, f.status, f.uploaded_at
    FROM uploaded_finance_files f
    WHERE f.file_type = 'tb_current'
      AND f.period_year = 2026
      AND f.status = 'processed'
      AND f.id = (
          SELECT max(f2.id)
          FROM uploaded_finance_files f2
          WHERE f2.file_type = 'tb_current'
            AND f2.period_year = f.period_year
            AND f2.period_month = f.period_month
            AND f2.status = 'processed'
      )
    ORDER BY f.id ASC;
'''
eligible_files = pd.read_sql_query(files_query, conn)
display(eligible_files)

file_ids = tuple(eligible_files['id'].tolist())
placeholders = ','.join(['?'] * len(file_ids))

# 2. Query mapped financial records
records_query = f'''
    SELECT 
        uploaded_file_id,
        period_year,
        period_month,
        revenue_category,
        period_activity
    FROM financial_tb_records
    WHERE uploaded_file_id IN ({placeholders})
      AND revenue_category IS NOT NULL
      AND trim(revenue_category) != '';
'''
raw_records = pd.read_sql_query(records_query, conn, params=file_ids)
conn.close()

print(f'Total Mapped Records Loaded: {len(raw_records):,}')

Cell executed successfully.


## 3. Monthly Revenue Calculation & Feature Engineering

In [3]:
# Apply standard sign convention: -sum(period_activity) / 1,000,000
raw_records['revenue'] = -raw_records['period_activity'].astype(float) / 1_000_000.0

monthly_category_df = (
    raw_records.groupby(['revenue_category', 'period_month', 'period_year'], as_index=False)['revenue']
    .sum()
    .rename(columns={'revenue': 'current_revenue'})
)
monthly_category_df['month_number'] = monthly_category_df['period_month'].map(MONTH_ORDER)
monthly_category_df['time_index'] = monthly_category_df['month_number'] - 1
monthly_category_df = monthly_category_df.sort_values(['revenue_category', 'month_number']).reset_index(drop=True)

# Compute 1-month lag of current year actual revenue
monthly_category_df['current_revenue_lag_1'] = monthly_category_df.groupby('revenue_category')['current_revenue'].shift(1)

# Drop Month 1 (January) from target observations; January acts as lag feature for February
model_data = monthly_category_df.dropna(subset=['current_revenue_lag_1']).reset_index(drop=True)

feature_columns = ['revenue_category', 'month_number', 'time_index', 'current_revenue_lag_1']
categorical_features = ['revenue_category']
numeric_features = ['month_number', 'time_index', 'current_revenue_lag_1']
target_column = 'current_revenue'

print(f'Valid Model Samples: {len(model_data)} across {model_data["revenue_category"].nunique()} categories')
display(model_data.head(10))

Cell executed successfully.


## 4. Chronological Train / Test Split & Pipeline Fit

In [4]:
available_months = sorted(model_data['month_number'].unique())
latest_month_num = max(available_months)

# Chronological Split: Train on earlier months, hold out latest month for testing
train_data = model_data[model_data['month_number'] < latest_month_num].copy()
test_data = model_data[model_data['month_number'] == latest_month_num].copy()

X_train = train_data[feature_columns]
y_train = train_data[target_column]
X_test = test_data[feature_columns]
y_test = test_data[target_column]

print(f'Training Samples: {len(X_train)} (Months: {train_data["period_month"].unique().tolist()})')
print(f'Testing Samples:  {len(X_test)} (Month:  {test_data["period_month"].unique().tolist()})')

# Pipeline
preprocessor = ColumnTransformer(
    transformers=[
        ('category', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore'))]), categorical_features),
        ('numeric', SimpleImputer(strategy='median'), numeric_features),
    ]
)

model = Pipeline(
    steps=[
        ('preprocessor', preprocessor),
        ('regressor', RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1)),
    ]
)

model.fit(X_train, y_train)
print('RandomForestRegressor model successfully fitted.')

Cell executed successfully.


## 5. Model Evaluation Metrics

In [5]:
test_preds = model.predict(X_test)
mae = mean_absolute_error(y_test, test_preds)
rmse = np.sqrt(mean_squared_error(y_test, test_preds))
r2 = r2_score(y_test, test_preds)

metrics_df = pd.DataFrame({
    'Metric': ['MAE (Mean Absolute Error)', 'RMSE (Root Mean Squared Error)', 'R² (Coefficient of Determination)'],
    'Value': [f'{mae:.4f} M LKR', f'{rmse:.4f} M LKR', f'{r2:.4f}']
})
display(metrics_df)

eval_comparison = test_data[['revenue_category', 'current_revenue']].copy()
eval_comparison['predicted_revenue'] = test_preds.round(4)
eval_comparison['absolute_error'] = (eval_comparison['current_revenue'] - eval_comparison['predicted_revenue']).abs().round(4)
eval_comparison['pct_error'] = ((eval_comparison['absolute_error'] / eval_comparison['current_revenue']) * 100).round(2)
display(eval_comparison.sort_values('current_revenue', ascending=False))

Cell executed successfully.


## 6. Multi-Step Recursive Future Forecasts (+3 Months)

In [6]:
categories = sorted(model_data['revenue_category'].unique())
latest_actuals = monthly_category_df[monthly_category_df['month_number'] == latest_month_num].set_index('revenue_category')['current_revenue'].to_dict()

forecast_rows = []
lag_tracker = latest_actuals.copy()
FORECAST_HORIZON = 3

for offset in range(1, FORECAST_HORIZON + 1):
    f_month_num = latest_month_num + offset
    f_month_name = f'Month +{offset} (Month {f_month_num})'
    step_features = pd.DataFrame({
        'revenue_category': categories,
        'month_number': f_month_num,
        'time_index': (latest_month_num - 1) + offset,
        'current_revenue_lag_1': [lag_tracker[cat] for cat in categories]
    })
    step_preds = model.predict(step_features[feature_columns])
    for cat, pred in zip(categories, step_preds):
        pred_val = round(float(pred), 4)
        forecast_rows.append({
            'period': f_month_name,
            'revenue_category': cat,
            'forecast_revenue': pred_val
        })
        lag_tracker[cat] = pred_val  # update lag recursively

forecast_pivot = pd.DataFrame(forecast_rows).pivot(index='revenue_category', columns='period', values='forecast_revenue')
display(forecast_pivot)

Cell executed successfully.
